# Tool Use with LLMs

Enabling language models to interact with external systems through function calling.

## What is Tool Use?

**Tool use** (also called **function calling**) allows LLMs to interact with external systems by:

1. Recognizing when a tool is needed to answer a query
2. Generating structured calls to that tool
3. Incorporating the tool's response into the final answer

This bridges the gap between the LLM's knowledge and real-world actions.

## Example Use Cases

| Tool | What it does | Example query |
|------|--------------|---------------|
| **Calculator** | Perform math operations | "What is 15% of 847?" |
| **Web Search** | Retrieve current information | "What's the weather in Berlin?" |
| **Database Query** | Look up structured data | "How many orders did customer X place?" |
| **API Call** | Interact with external services | "Send a Slack message to the team" |
| **Code Execution** | Run Python code | "Plot the sales data from this CSV" |

## Setup

In [ ]:
from llama_index.llms.ollama import Ollama
from llama_index.core.tools import FunctionTool
from IPython.display import Markdown
from ai_dojo.lm import get_ollama_url

# Connect to Ollama
llm = Ollama(
    model="qwen3.5:4b",
    base_url=get_ollama_url(),
    request_timeout=120.0,
    context_window=8192,  # or 4096 for even less memory
    additional_kwargs={
        "num_ctx": 8192,  # Pass to Ollama directly
    }
)

## Defining Tools with LlamaIndex

A tool in LlamaIndex needs:
1. A Python function that performs the action
2. A description that helps the LLM understand when to use it

In [ ]:
# Example: A simple calculator tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers and return the result."""
    return a * b

def add(a: float, b: float) -> float:
    """Add two numbers and return the result."""
    return a + b

# Wrap as LlamaIndex tools
multiply_tool = FunctionTool.from_defaults(fn=multiply)
add_tool = FunctionTool.from_defaults(fn=add)

# Test the tools via their .call interface
multiply_tool.call(a=7, b=6), add_tool.call(a=10, b=32)

## Example: Date/Time Tool

In [ ]:
from datetime import datetime

def get_current_datetime() -> str:
    """Get the current date and time. Use this when the user asks about today's date or current time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

datetime_tool = FunctionTool.from_defaults(fn=get_current_datetime)

# Test it
datetime_tool.call()

## Using Tools with an LLM



In [ ]:
from llama_index.core.agent.workflow import ReActAgent

agent = ReActAgent(
    tools=[multiply_tool, add_tool, datetime_tool],
    llm=llm,
    verbose=True
)



In [ ]:
from llama_index.core.workflow import Context


# Create a context for the conversation
ctx = Context(agent)

# Use await with .run() instead of .chat()
# (In Jupyter, you can use await directly)
response = await agent.run(user_msg="What is 25 multiplied by 42 multiplied by 13?", ctx=ctx)

Markdown(f"**Answer:** {response}")

In [ ]:
from rich import print as rprint

In [ ]:
rprint(response)

In [ ]:
response = await agent.run(
    user_msg="""A store sells notebooks for €3.10 each and pens for €2.50 each. 
    Maria buys 15 notebooks and 20 pens. How much does she pay in total?""",
    ctx=ctx
)
Markdown(f"**Answer:** {response}")

In [ ]:
rprint(response)

<div style="border:2px solid gray; padding:10px; width: 95%;">
<b>🫳 Hands-on: Solve a math problem with tool-using agents</b>

A warehouse manager has a budget of €125,000. She orders 127 pallets of supplies at €289 each and 83 crates of equipment at €156 each. She divides the remaining budget equally among her 17 employees as a year-end bonus. **How much does each employee receive?**

The numbers in this problem are deliberately large to exceed the typical arithmetic ability of LLMs — they need tools to get this right!

**Your task:**
1. Add the missing tool functions 
2. Configure a new agent with all required tools
3. Run the agent on the problem and verify the answer

</div>

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2025-2026 [Christian Staudt](https://clstaudt.me) and [Katharina Rasch](https://krasch.io)_